# Workstream 2: LitCoin Embedding And Retrieval Comparison

This notebook compares OpenAI and PubMedBERT SapBERT relationship embeddings for the same LitCoin relationship edges. It uses tested helper functions from `analysis.embedding_comparison` so that validation, projections, metrics, and exports are reproducible outside notebook cell state.

Projection caveat: OpenAI and SapBERT embeddings occupy different vector spaces and may have different dimensions. The panels below are fitted separately. Compare highlighted edge membership, nearest-neighbor agreement, publication concentration, and predicate/category patterns, not absolute axis direction, rotation, or inter-panel coordinates.

In [1]:
import os
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import IFrame, display
from tqdm import TqdmWarning

from analysis.embedding_comparison import (
    add_query_projection_marker,
    apply_query_highlights,
    build_embedding_collection,
    build_retrieval_comparison_table,
    compare_query_neighborhoods,
    comparison_manifest,
    deduplicate_exact_duplicate_rows,
    load_retrieval_result_sets_from_path_search_cache,
    match_embedding_collections,
    nearest_neighbor_agreement_rows,
    nearest_neighbor_jaccard,
    nearest_neighbors,
    project_embeddings,
    read_jsonl_rows,
    same_metadata_fraction,
    summarize_anchor_diversity,
    write_json,
    write_neighbor_agreement_html,
    write_projection_html,
    write_rows_csv,
)
from src.embeddings.embedding_utils import get_embedding_client

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "scripts" / "data"
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "outputs" / "embedding_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=TqdmWarning, message="IProgress not found.*")

OPENAI_RELATIONSHIP_EXPORT = DATA_DIR / "relationship_embeddings.jsonl"
SAPBERT_RELATIONSHIP_EXPORT = DATA_DIR / "sapbert_relationship_embeddings.jsonl"
RANDOM_SEED = 13
TOP_K = 10
PROJECTION_METHOD = "pca"  # Use "umap" for exploratory local-neighborhood views.
COLOR_FIELD = "predicate"  # Try: predicate_family, relationship_kind, is_mentions_edge, endpoint_prefix_pair, endpoint_label_pair, publication_id

QUERIES = [
    "drug resistance in cancer",
    "genes involved in chemoresistance in cancer",
    "PTEN cancer chemoresistance",
    "therapeutic response relationships",
]

def display_html_artifact(path: Path, *, height: int = 760) -> IFrame:
    relative = path.resolve().relative_to(PROJECT_ROOT.resolve())
    return IFrame(src=f"/files/{relative.as_posix()}", width="100%", height=height)

## Load And Validate Exports

The JSONL exports should contain one row per relationship per model. The strict loader fails on missing IDs, duplicate IDs within one model export, and inconsistent dimensions. The current local exports contain exact duplicate relationship rows from an older undirected dump query, so this notebook removes only exact duplicate rows and records that normalization in the manifest. Embedding arrays are retained in memory only and are removed from exported tables and figure hover payloads.

In [2]:
openai_rows, openai_normalization = deduplicate_exact_duplicate_rows(
    read_jsonl_rows(OPENAI_RELATIONSHIP_EXPORT), model_name="openai"
)
sapbert_rows, sapbert_normalization = deduplicate_exact_duplicate_rows(
    read_jsonl_rows(SAPBERT_RELATIONSHIP_EXPORT), model_name="sapbert"
)
openai_edges = build_embedding_collection(openai_rows, model_name="openai", embedding_key="embedding")
sapbert_edges = build_embedding_collection(sapbert_rows, model_name="sapbert", embedding_key="sapbert_embedding")
matched = match_embedding_collections(openai_edges, sapbert_edges)

manifest = comparison_manifest(
    collections=[openai_edges, sapbert_edges],
    projection_parameters={"method": "pca", "random_seed": RANDOM_SEED},
    queries=QUERIES,
    seed=RANDOM_SEED,
)
manifest["coverage"] = matched.coverage_report
manifest["input_normalization"] = {"openai": openai_normalization, "sapbert": sapbert_normalization}
write_json(manifest, OUTPUT_DIR / "manifest.json")
matched.coverage_report

{'left_model': 'openai',
 'right_model': 'sapbert',
 'left_count': 2003,
 'right_count': 2003,
 'matched_count': 2003,
 'left_only_count': 0,
 'right_only_count': 0,
 'left_only_ids': [],
 'right_only_ids': [],
 'same_population': True}

## Side-By-Side Projections

PCA is used here as a deterministic reference projection. UMAP can be requested with `PROJECTION_METHOD = "umap"` for exploratory local-neighborhood views. The generated HTML uses click-to-inspect details instead of hover tooltips.

In [3]:
PROJECTION_METHOD = "umap"
openai_projection = project_embeddings(openai_edges, relationship_ids=matched.relationship_ids, method=PROJECTION_METHOD, seed=RANDOM_SEED)
sapbert_projection = project_embeddings(sapbert_edges, relationship_ids=matched.relationship_ids, method=PROJECTION_METHOD, seed=RANDOM_SEED)

projection_rows = openai_projection.rows + sapbert_projection.rows
write_rows_csv(projection_rows, OUTPUT_DIR / f"{PROJECTION_METHOD}_projection_rows.csv")
projection_html_path = write_projection_html(
    [openai_projection, sapbert_projection],
    OUTPUT_DIR / f"{PROJECTION_METHOD}_projection.html",
    color_field=COLOR_FIELD,
)

display_html_artifact(projection_html_path)

/home/hongyi/.cache/uv/archive-v0/ofEju-4zG69Z-ABSNjyXf/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/hongyi/.cache/uv/archive-v0/ofEju-4zG69Z-ABSNjyXf/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## Query Neighborhood Explorer

Set `TEST_QUERY` and rerun this cell to compare the closest relationship embeddings for OpenAI and SapBERT. The highlighted projection marks each model's top-k query neighbors with numbered open circles, and the red `Q` marker shows the projected input-query embedding in each model's own vector space. This requires embedding the query text for both models; OpenAI requires a configured API key, while SapBERT uses the local model path/configuration.

In [4]:
TEST_QUERY = "genes involved in chemoresistance in cancer"
QUERY_TOP_K = 15

openai_query_embedding = get_embedding_client("openai").embed_query(TEST_QUERY)
sapbert_query_embedding = get_embedding_client("sapbert").embed_query(TEST_QUERY)

query_comparison = compare_query_neighborhoods(
    query=TEST_QUERY,
    left=openai_edges,
    right=sapbert_edges,
    left_query_embedding=openai_query_embedding,
    right_query_embedding=sapbert_query_embedding,
    k=QUERY_TOP_K,
    relationship_ids=matched.relationship_ids,
)

openai_query_rows = add_query_projection_marker(
    apply_query_highlights(
        openai_projection.rows,
        query_comparison.left_results,
        top_k=QUERY_TOP_K,
        query=TEST_QUERY,
        label="OpenAI query neighbors",
    ),
    openai_edges,
    openai_query_embedding,
    query=TEST_QUERY,
    seed=RANDOM_SEED,
)
sapbert_query_rows = add_query_projection_marker(
    apply_query_highlights(
        sapbert_projection.rows,
        query_comparison.right_results,
        top_k=QUERY_TOP_K,
        query=TEST_QUERY,
        label="SapBERT query neighbors",
    ),
    sapbert_edges,
    sapbert_query_embedding,
    query=TEST_QUERY,
    seed=RANDOM_SEED,
)

query_projection_html_path = write_projection_html(
    [{"rows": openai_query_rows}, {"rows": sapbert_query_rows}],
    OUTPUT_DIR / "query_projection.html",
    color_field=COLOR_FIELD,
    title=f"{PROJECTION_METHOD.upper()} projection with top-{QUERY_TOP_K} query neighbors: {TEST_QUERY}",
)
write_rows_csv(query_comparison.left_results, OUTPUT_DIR / "openai_query_neighbors.csv")
write_rows_csv(query_comparison.right_results, OUTPUT_DIR / "sapbert_query_neighbors.csv")
write_json(
    {
        "query": TEST_QUERY,
        "top_k": QUERY_TOP_K,
        "overlap": query_comparison.overlap,
        "rank_correlation": query_comparison.rank_correlation,
    },
    OUTPUT_DIR / "query_neighbor_comparison.json",
)

display_columns = [
    "rank",
    "relationship_id",
    "similarity",
    "predicate",
    "predicate_family",
    "subject",
    "object",
    "publication_id",
]
print("Top-k overlap:", query_comparison.overlap)
print("Shared-candidate rank correlation:", query_comparison.rank_correlation)
display(pd.DataFrame(query_comparison.left_results)[display_columns].style.set_caption("OpenAI nearest relationships"))
display(pd.DataFrame(query_comparison.right_results)[display_columns].style.set_caption("SapBERT nearest relationships"))
display_html_artifact(query_projection_html_path)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3525.35it/s]
/home/hongyi/.cache/uv/archive-v0/ofEju-4zG69Z-ABSNjyXf/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/hongyi/.cache/uv/archive-v0/ofEju-4zG69Z-ABSNjyXf/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Top-k overlap: {'k': 15, 'left_ids': ['95eb9fcae394a691', '925e8bddcf9141bc', '2fb96d28fc79f25c', '0d7f58617dad9fc4', '487a4888b064dc2c', '2a61749591fd2113', '871d44f652c1a1f0', 'b8dc138ace7d4842', '635f087056d54a1c', '1bdd04566ad37675', '3a67a7734219e815', '92a637f1d8fc8f75', '94db0ac5a94fb177', 'cd3f0f41b1f8b345', '0ee7f69ebd71d59d'], 'right_ids': ['246d459ac0ee1026', '95eb9fcae394a691', '6362d2ef412961f9', '925e8bddcf9141bc', '1bdd04566ad37675', 'a37681c93330c750', 'b8dc138ace7d4842', '635f087056d54a1c', '3a67a7734219e815', '554377d0508daa3e', '6e79f0b1798e6939', '21273a8ed3206bb8', '92a637f1d8fc8f75', 'bf9daabc78e54634', '4804545c604afc87'], 'shared_ids': ['1bdd04566ad37675', '3a67a7734219e815', '635f087056d54a1c', '925e8bddcf9141bc', '92a637f1d8fc8f75', '95eb9fcae394a691', 'b8dc138ace7d4842'], 'intersection_count': 7, 'union_count': 23, 'jaccard': 0.30434782608695654}
Shared-candidate rank correlation: {'shared_count': 7, 'spearman': 0.8402898592489522, 'shared_ids': ['1bdd04566ad

,rank,relationship_id,similarity,predicate,predicate_family,subject,object,publication_id
0,1,95eb9fcae394a691,0.489602,biolink:mentions,unknown,PMID:35204148,UMLS:C0920425,None
1,2,925e8bddcf9141bc,0.466010,biolink:mentions,unknown,PMID:35204148,CHEBI:26523,None
2,3,2fb96d28fc79f25c,0.452178,biolink:treats,treatment,REACT:R-MMU-1296067,NCBIGene:309902,PMID:35304127
3,4,0d7f58617dad9fc4,0.451354,biolink:mentions,unknown,PMID:20126413,UMLS:C0017361,None
4,5,487a4888b064dc2c,0.449174,biolink:mentions,unknown,PMID:32694383,UMLS:C0762890,None
5,6,2a61749591fd2113,0.448408,biolink:mentions,unknown,PMID:16410744,NCIT:C132226,None
6,7,871d44f652c1a1f0,0.445167,biolink:mentions,unknown,PMID:20126413,NCBIGene:5728,None
7,8,b8dc138ace7d4842,0.444860,biolink:mentions,unknown,PMID:35204148,UMLS:C0808232,None
8,9,635f087056d54a1c,0.443024,biolink:mentions,unknown,PMID:35204148,UMLS:C3873567,None
9,10,1bdd04566ad37675,0.440945,biolink:mentions,unknown,PMID:35204148,UMLS:C3178870,None


,rank,relationship_id,similarity,predicate,predicate_family,subject,object,publication_id
0,1,246d459ac0ee1026,0.483417,biolink:coexists_with,related,CHEBI:26523,UMLS:C3873567,PMID:35204148
1,2,95eb9fcae394a691,0.482741,biolink:mentions,unknown,PMID:35204148,UMLS:C0920425,None
2,3,6362d2ef412961f9,0.469317,biolink:coexists_with,related,UMLS:C5677936,UMLS:C3873567,PMID:35204148
3,4,925e8bddcf9141bc,0.469054,biolink:mentions,unknown,PMID:35204148,CHEBI:26523,None
4,5,1bdd04566ad37675,0.467107,biolink:mentions,unknown,PMID:35204148,UMLS:C3178870,None
5,6,a37681c93330c750,0.464417,biolink:correlated_with,association,UMLS:C0814299,MONDO:0005530,PMID:34702274
6,7,b8dc138ace7d4842,0.460880,biolink:mentions,unknown,PMID:35204148,UMLS:C0808232,None
7,8,635f087056d54a1c,0.459003,biolink:mentions,unknown,PMID:35204148,UMLS:C3873567,None
8,9,3a67a7734219e815,0.457319,biolink:mentions,unknown,PMID:35204148,UMLS:C5677936,None
9,10,554377d0508daa3e,0.450040,biolink:mentions,unknown,PMID:35204148,NCBITaxon:1407750,None


## Neighborhood Agreement

These metrics compare local neighborhood membership within each embedding model. They do not compare raw coordinate values across model spaces.

In [ ]:
openai_neighbors = nearest_neighbors(openai_edges, k=TOP_K, relationship_ids=matched.relationship_ids)
sapbert_neighbors = nearest_neighbors(sapbert_edges, k=TOP_K, relationship_ids=matched.relationship_ids)

neighbor_agreement = nearest_neighbor_jaccard(openai_neighbors, sapbert_neighbors, k=TOP_K)
openai_publication_fraction = same_metadata_fraction(openai_edges, openai_neighbors, metadata_field="publication_id", k=TOP_K)
sapbert_publication_fraction = same_metadata_fraction(sapbert_edges, sapbert_neighbors, metadata_field="publication_id", k=TOP_K)
openai_predicate_fraction = same_metadata_fraction(openai_edges, openai_neighbors, metadata_field="predicate_family", k=TOP_K)
sapbert_predicate_fraction = same_metadata_fraction(sapbert_edges, sapbert_neighbors, metadata_field="predicate_family", k=TOP_K)

neighbor_metrics = {
    "top_k": TOP_K,
    "nearest_neighbor_jaccard": neighbor_agreement,
    "openai_same_publication_fraction": openai_publication_fraction,
    "sapbert_same_publication_fraction": sapbert_publication_fraction,
    "openai_same_predicate_family_fraction": openai_predicate_fraction,
    "sapbert_same_predicate_family_fraction": sapbert_predicate_fraction,
}
write_json(neighbor_metrics, OUTPUT_DIR / "nearest_neighbor_metrics.json")

agreement_rows = nearest_neighbor_agreement_rows(openai_edges, neighbor_agreement)
write_rows_csv(agreement_rows, OUTPUT_DIR / "nearest_neighbor_agreement_rows.csv")
agreement_html_path = write_neighbor_agreement_html(
    agreement_rows,
    OUTPUT_DIR / "nearest_neighbor_agreement.html",
    title=f"OpenAI/SapBERT top-{TOP_K} nearest-neighbor agreement",
)

summary_rows = [
    {"metric": "top_k", "value": TOP_K},
    {"metric": "mean_neighbor_jaccard", "value": neighbor_agreement["mean_jaccard"]},
    {"metric": "openai_same_publication_mean", "value": openai_publication_fraction["mean_fraction"]},
    {"metric": "sapbert_same_publication_mean", "value": sapbert_publication_fraction["mean_fraction"]},
    {"metric": "openai_same_predicate_family_mean", "value": openai_predicate_fraction["mean_fraction"]},
    {"metric": "sapbert_same_predicate_family_mean", "value": sapbert_predicate_fraction["mean_fraction"]},
]
display(pd.DataFrame(summary_rows))
display(
    pd.DataFrame(agreement_rows)
    .sort_values(["neighbor_jaccard", "relationship_id"])
    .head(20)[["relationship_id", "neighbor_jaccard", "predicate_family", "subject", "object", "publication_id"]]
    .style.set_caption("Lowest-agreement relationships")
)
display_html_artifact(agreement_html_path, height=620)

## Retrieval Result Comparison

This cell loads retrieval diagnostics from the explorer path-search cache, whose default location is `/tmp/kg_explorer/path_search_cache.json` unless `KG_EXPLORER_PATH_CACHE` points elsewhere. Run the desired searches once for each model/retrieval mode you want to compare, then rerun this cell. Raw dense/keyword/hybrid candidates, reranked candidates, and final anchors are kept as separate result-set types.

In [ ]:
PATH_SEARCH_CACHE = Path(os.getenv("KG_EXPLORER_PATH_CACHE", "/tmp/kg_explorer/path_search_cache.json"))
retrieval_result_sets = load_retrieval_result_sets_from_path_search_cache(PATH_SEARCH_CACHE)
retrieval_table = build_retrieval_comparison_table(retrieval_result_sets)
write_rows_csv(retrieval_table, OUTPUT_DIR / "retrieval_comparison.csv")

diversity_rows = [
    summarize_anchor_diversity(group.to_dict("records"), query=query)
    for query, group in pd.DataFrame(retrieval_table).groupby("query")
] if retrieval_table else []
write_rows_csv(diversity_rows, OUTPUT_DIR / "anchor_diversity_summary.csv")

if retrieval_table:
    display(pd.DataFrame(retrieval_table).head(20))
    display(pd.DataFrame(diversity_rows))
else:
    print(f"No retrieval diagnostics found in {PATH_SEARCH_CACHE}. Run explorer searches first or set PATH_SEARCH_CACHE to a saved cache JSON file.")

## Conclusions Template

Direct observations:

- Record matched edge coverage, dimensions, nearest-neighbor agreement, and retrieval overlaps from the generated tables.
- Identify whether top results concentrate in a small number of publications or predicate families.
- Compare raw dense candidates with reranked/diversified anchors separately.

Interpretations:

- Explain whether differences appear useful for graph exploration entry points rather than claiming global model superiority.
- Connect examples to the path-centric workflow: Search -> Candidate Paths -> Human Selection -> Expansion.

Future hypotheses:

- Candidate retrieval may benefit from model-specific routing or hybrid weighting, but only after evaluating more queries and human judgments.
- A small diagnostic panel may be useful later if it explains selected anchors without turning the Dash app into an embedding browser.